In [3]:
# main.py
%load_ext autoreload
%autoreload 2

import sys
import subprocess
import time
sys.path.insert(0,"../data_fetch")
sys.path.insert(0,"../pre_processing")
sys.path.insert(0,"../triple_store_ingestion")
sys.path.insert(0,"../RDF2TSS_V2")
sys.path.insert(0,"../RDF2LDES")


list

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


list

# Step 1: Fetch the data


In [56]:
import fetch

In [57]:
fetch.fetch_stations()
fetch.fetch_timeseries()

Station fetching started.
Station fetching finished & file saved.
Timeseries fetching started.
Timeseries fetching finished & file saved.


# Step 2: Pre-Process

In [58]:
import preprocess

In [59]:
preprocess.preprocess()

                   Timestamp    Value  Interpolation Type  Quality Code  \
0  2021-03-03 23:15:00+00:00  3606.54                 104           100   
1  2021-03-03 23:45:00+00:00  3556.45                 104           100   
2  2021-03-04 00:30:00+00:00  3593.75                 104           100   
3  2021-03-04 00:45:00+00:00  3588.33                 104           100   
4  2021-03-04 01:00:00+00:00  3587.95                 104           100   

  Quality Code Name      Quality Code Description    Quality Code Color  \
0           GoodExt  Good quality: external - 100  rgba(0, 128, 0, 1.0)   
1           GoodExt  Good quality: external - 100  rgba(0, 128, 0, 1.0)   
2           GoodExt  Good quality: external - 100  rgba(0, 128, 0, 1.0)   
3           GoodExt  Good quality: external - 100  rgba(0, 128, 0, 1.0)   
4           GoodExt  Good quality: external - 100  rgba(0, 128, 0, 1.0)   

   Aggregation Accuracy %  Absolute Value  AV Interpolation Type  ...  \
0                     NaN

# step 3 RML-Mapping

In [60]:
command = [
    "java", 
    "-jar", "rmlmapper.jar", 
    "-m", "./RML_mapping/timeseriesmapping.rml.ttl", 
    "-o", "../data/timeseries.ttl", 
    "-s", "turtle"
]

try:
    # capture_output=True grabs stdout and stderr
    # text=True ensures the output is returned as a string, not bytes
    result = subprocess.run(command, capture_output=True, text=True, check=True)
    
    print("RML Mapping completed successfully.")
    print(result.stdout)

except subprocess.CalledProcessError as e:
    print(f"The command failed with error code: {e.returncode}")
    print("-" * 20)
    print("STDOUT (Standard Output):")
    print(e.stdout)
    print("-" * 20)
    print("STDERR (The actual error message):")
    print(e.stderr)  # This is where the Java stack trace will appear

RML Mapping completed successfully.



# step 4: Ingest to Virtuoso

In [61]:
import ingest

In [62]:
ttl_timeseries_path = "../data/timeseries.ttl"
ttl_stations_path = "../data/stations.ttl"
GRAPH_URI = "http://example.com/Gent-Terneuzen"

In [63]:
ingest.delete_graph(GRAPH_URI)
ingest.upload_graph(ttl_timeseries_path, GRAPH_URI)
ingest.upload_graph(ttl_stations_path, GRAPH_URI)

Attempting to delete graph: http://example.com/Gent-Terneuzen...


Successfully deleted graph: http://example.com/Gent-Terneuzen
started uploading ../data/timeseries.ttl to http://example.com/Gent-Terneuzen
Successfully uploaded ../data/timeseries.ttl to http://example.com/Gent-Terneuzen
started uploading ../data/stations.ttl to http://example.com/Gent-Terneuzen
Successfully uploaded ../data/stations.ttl to http://example.com/Gent-Terneuzen


# step 5: RDF2TSS

In [64]:
import RDF2TSS_V2

In [65]:
input_path = "../data/timeseries.ttl"
output_path = "../data/TSSgraph.ttl"


In [66]:
original_graph = RDF2TSS_V2.load_graph(input_path)
sensor_set = RDF2TSS_V2.create_sensor_set(original_graph)
tss_graph = RDF2TSS_V2.create_tss(sensor_set, original_graph)
RDF2TSS_V2.save_graph(output_path, tss_graph)

Started loading graph from ../data/timeseries.ttl...
Graph loaded successfully.
Started identifying unique sensors...
Identified sensor: http://example.com/waterinfo/289423042
Identified sensor: http://example.com/waterinfo/289429042
Identified sensor: http://example.com/waterinfo/289435042
Identified sensor: http://example.com/waterinfo/289441042
4 Sensors identified successfully.
Started extracting observations per sensor...
Processing sensor 1/4
Processing sensor 2/4
Processing sensor 3/4
Processing sensor 4/4
TSS graph creation complete.
Started writing file to disk: ../data/TSSgraph.ttl
File written successfully.


# step 6 (Optional) Ingest RDF TSS into Virtuoso

In [67]:
import ingest

In [68]:
RDFTSS_path = "../data/TSSgraph.ttl"
TSS_GRAPH_URI = "http://example.com/Gent-Terneuzen-TSS"

In [69]:
ingest.delete_graph(TSS_GRAPH_URI)
ingest.upload_graph(RDFTSS_path, TSS_GRAPH_URI)


Attempting to delete graph: http://example.com/Gent-Terneuzen-TSS...
Successfully deleted graph: http://example.com/Gent-Terneuzen-TSS
started uploading ../data/TSSgraph.ttl to http://example.com/Gent-Terneuzen-TSS


Failed to upload. Status code: 500
Response: <!DOCTYPE HTML PUBLIC "-//IETF//DTD HTML 2.0//EN">
<html>
  <head>
    <title>SPARQL Graph Protocol request failed</title>
  </head>
  <body>
    <h1>HTTP/1.1 500 Request Failed</h1>
    <h3>Error 37000</h3>
<xmp>[Vectorized Turtle loader] TURTLE RDF loader, line 424710: SP029: TURTLE RDF loader, line 424710: The triple-double-quoted string is too long at """[
 {
  "id": "https://example.org/tss/snippet/reading/2",
  "time": "2021-03-03T23:45:00+00:00",
  "value": 902.27
 },
 {
  "id": "https://example.org/tss/snippet/reading/2",
  "time": "2021-03-04T00:15:00+00:00",
  "value": 901.96
 },
 {
  "id": "https://example.org/tss/snippet/reading/2",
  "time": "2021-03-04T01:30:00+00:00",
  "value": 901.08
 },
 {
  "id": "https://example.org/tss/snippet/reading/2",
  "time": "2021-03-04T01:45:00+00:00",
  "value": 906.35
 },
 {
  "id": "https://example.org/tss/snippet/reading/2",
  "time": "2021-03-04T04:15:00+00:00",
  "value": 906.05
 },
 {
  "i

# step 7: transoform data to LDES

In [11]:
import RDFTSS2LDES

In [ ]:
input_path = "../data/TSSgraph.ttl"

In [9]:
print("Starting processing...")
start_time = time.perf_counter()
original_graph = RDFTSS2LDES.load_graph(input_path)
result = RDFTSS2LDES.process_graph(original_graph)
RDFTSS2LDES.divide_data(result)
end_time = time.perf_counter()
print(f"Processing completed in {end_time - start_time:.2f} seconds.")

Starting processing...
Total snippets processed: 4
Processing completed in 48.84 seconds.


In [10]:
start_time = time.perf_counter()
RDFTSS2LDES.delete_log()
RDFTSS2LDES.delete_ldes_files()
RDFTSS2LDES.create_ldes_files()
end_time = time.perf_counter()
print(f"Processing completed in {end_time - start_time:.2f} seconds.")

Processing completed in 3.94 seconds.
